# Data cleaning — seven-zone hourly panel

Builds an hourly panel of day-ahead prices and forecasts for seven bidding zones:
`DK1`, `DK2`, `NL`, `DE_LU`, `NO2`, `SE3`, `SE4`.

Three steps: read those zones from the raw ENTSO-E dump, collapse every series to
the clock hour, then measure what is missing.

Everything works in **UTC**, where every day has 24 hours and DST does not exist,
so a hole is a hole in what the platform published. Local market time is a later
concern.

## 1. Load

Reads the day-ahead price, load forecast and wind & solar generation forecast for
the seven zones, up to and including the hour beginning **2025-09-30 22:00 UTC**.
The zone list, the reservoir exclusion and the end of the window are applied as
predicates on the parquet scan, so only those rows are read.

The window ends there because the day-ahead market time unit changes to 15 minutes
on 2025-10-01. The bound covers that final hour in full — its `PT15M` slots are
read too, so it collapses from four quarters rather than from one.

Water reservoir & hydro storage [16.1.D] is excluded: it is published weekly
(`P7D`) as a stock level, not per market time unit as a flow.

Three assertions guard the read — every requested zone is defined in
`entsoe_tp.areas`, every requested zone returned rows, and nothing past the window
was read.

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 250)
pd.set_option("display.max_columns", 60)

# Resolve the raw dump relative to the notebook, wherever it is opened from.
CANDIDATES = [
    Path("datasets/nordic_baltic_raw.parquet"),
    Path("../datasets/nordic_baltic_raw.parquet"),
    Path("../../datasets/nordic_baltic_raw.parquet"),
]
RAW = next((p for p in CANDIDATES if p.exists()), None)
if RAW is None:
    raise FileNotFoundError(
        "nordic_baltic_raw.parquet not found. Set RAW manually to its path."
    )

# Zone timezones come from the package that produced the file, so the two cannot
# drift apart.
sys.path.insert(0, str(RAW.parent.parent))
from entsoe_tp.areas import AREAS, lookup  # noqa: E402

# --- what gets read -----------------------------------------------------------
# The zones covered, the variable left out, and the last hour taken. All three
# become predicates on the parquet scan in load_raw().
ZONES = ["DK1", "DK2", "NL", "DE_LU", "NO2", "SE3", "SE4"]
EXCLUDE_VARIABLES = ["reservoir"]

# Last hour read, labelled by its start and included in full. The day-ahead market
# time unit changes to 15 minutes on 2025-10-01.
END_UTC = pd.Timestamp("2025-09-30 22:00", tz="UTC")

unknown = sorted(set(ZONES) - set(AREAS))
assert not unknown, f"not defined in entsoe_tp.areas: {unknown}"

# psrType codes carried by the wind & solar forecast document (A69), spelled out at
# load time so every table below reads in words rather than codes. Unlisted codes
# pass through as themselves, so a new production type stays visible.
PSR_NAMES = {"B16": "solar", "B18": "wind_offshore", "B19": "wind_onshore"}

KEY = ["zone", "variable", "psr_type", "resolution", "timestamp_utc"]
CAT = ["zone", "eic", "variable", "document_type", "process_type", "resolution",
       "psr_type", "business_type", "curve_type", "contract_type", "unit", "currency"]


def load_raw(path=RAW, zones=ZONES, exclude=EXCLUDE_VARIABLES,
             end=END_UTC) -> pd.DataFrame:
    """Read the rows this pipeline uses, and normalise the dictionary-encoded columns.

    ``zones``, ``exclude`` and ``end`` are pushed into the parquet scan as row
    filters, so only the matching rows are materialised.

    Nulls become empty strings on purpose: pandas treats NaN keys as *equal* in
    duplicated()/drop_duplicates(), which silently collapses every price row
    (psr_type is null for prices) into a single group. Empty strings behave correctly.
    """
    filters = [("zone", "in", list(zones))]
    if exclude:
        filters.append(("variable", "not in", list(exclude)))
    if end is not None:
        # The bound covers the whole hour `end` labels, sub-hourly slots included,
        # so that hour collapses from a full set of quarters rather than its first.
        filters.append(("timestamp_utc", "<", end + pd.Timedelta(hours=1)))

    df = pq.read_table(path, filters=filters).to_pandas()
    for c in CAT:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("").astype(str)
    df["psr_type"] = df["psr_type"].replace(PSR_NAMES)
    df["series"] = (
        df["zone"] + " | " + df["variable"]
        + np.where(df["psr_type"] != "", " [" + df["psr_type"] + "]", "")
    )
    # Zone order follows ZONES rather than the alphabet, so every table below reads
    # in the order the thesis lists them.
    df["zone"] = pd.Categorical(df["zone"], categories=list(zones), ordered=True)
    return df


raw = load_raw()

# A zone defined in areas.py but never captured by the dump would otherwise pass
# silently as an empty slice.
absent = [z for z in ZONES if not (raw.zone == z).any()]
assert not absent, f"no rows in the dump for {absent} -- has raw_dump run for them?"
assert not set(raw.variable) & set(EXCLUDE_VARIABLES), "an excluded variable was read"
assert raw.timestamp_utc.max() < END_UTC + pd.Timedelta(hours=1), "a stamp past END_UTC"

print(f"file      : {RAW.resolve()}")
print(f"zones     : {', '.join(ZONES)}")
print(f"not read  : {', '.join(EXCLUDE_VARIABLES) if EXCLUDE_VARIABLES else '(nothing)'}")
print(f"last hour : {END_UTC}")
print(f"{len(raw):,} rows x {raw.shape[1]} columns | {raw.series.nunique()} series")
print(f"span      : {raw.timestamp_utc.min()} -> {raw.timestamp_utc.max()} (UTC)")
print(f"variables : {', '.join(sorted(raw.variable.unique()))}")

display(raw.groupby(["zone", "variable", "psr_type"], observed=True)
           .agg(rows=("value", "size"),
                resolutions=("resolution", lambda s: ",".join(sorted(s.unique()))),
                first=("timestamp_utc", "min"),
                last=("timestamp_utc", "max")))

## 2. Duplicate rows

Two rows sharing the full key — zone, variable, production type, resolution and
timestamp — are two copies of one observation. Downloads are chunked by month with
inclusive endpoints, so consecutive chunks overlap at the join.

Counted first, dropped second. The analysis reports how many copies each key
carries, whether those copies agree on their value, and where the duplicated
stamps sit in the calendar month. A key whose copies disagree is one where
`keep="first"` selects between different numbers.

In [2]:
dup_any = raw.duplicated(KEY, keep=False)
redundant = int(raw.duplicated(KEY, keep="first").sum())

print(f"rows on a duplicated key: {int(dup_any.sum()):,} of {len(raw):,}")
print(f"redundant copies        : {redundant:,}")
print(f"keys carrying them      : {int(dup_any.sum()) - redundant:,}")

if not redundant:
    print("\nno duplicate keys")
else:
    # `series` encodes zone, variable and psr_type, so this grouping is the full key.
    per_key = (raw.loc[dup_any]
                  .groupby(["series", "resolution", "timestamp_utc"], observed=True)["value"]
                  .agg(copies="size", distinct="nunique", lo="min", hi="max"))
    per_key["spread"] = per_key["hi"] - per_key["lo"]
    pk = per_key.reset_index()

    print("\ncopies per duplicated key:")
    display(pk.copies.value_counts().sort_index()
              .rename_axis("copies").to_frame("keys"))

    # A key whose copies disagree makes keep="first" a choice rather than a formality.
    conflicting = pk[pk.distinct > 1]
    print(f"keys whose copies disagree on value: {len(conflicting):,} of {len(pk):,}")
    if len(conflicting):
        print("largest disagreements:")
        display(conflicting.nlargest(20, "spread").reset_index(drop=True))

    print("\nper series:")
    display(pk.groupby("series", observed=True)
              .agg(duplicated_stamps=("timestamp_utc", "size"),
                   redundant_copies=("copies", lambda c: int((c - 1).sum())),
                   disagreeing=("distinct", lambda x: int((x > 1).sum())),
                   max_spread=("spread", "max"),
                   first=("timestamp_utc", "min"),
                   last=("timestamp_utc", "max")))

    print("\nwhere the duplicated stamps sit in the month:")
    display(pk.timestamp_utc.dt.day.value_counts().sort_index()
              .rename_axis("day_of_month").to_frame("duplicated_stamps"))

rows on a duplicated key: 56,304 of 4,722,140
redundant copies        : 28,248
keys carrying them      : 28,056

copies per duplicated key:


,keys
copies,
2,27960
4,96


keys whose copies disagree on value: 192 of 28,056
largest disagreements:


,series,resolution,timestamp_utc,copies,distinct,lo,hi,spread
0,DE_LU | price,PT15M,2025-10-01 16:15:00+00:00,4,2,200.00,478.69,278.69
1,DE_LU | price,PT15M,2025-10-01 17:45:00+00:00,4,2,253.14,517.12,263.98
2,DE_LU | price,PT15M,2025-10-01 17:30:00+00:00,4,2,321.94,517.12,195.18
3,DE_LU | price,PT15M,2025-10-01 05:30:00+00:00,4,2,180.23,326.64,146.41
4,DE_LU | price,PT15M,2025-10-01 17:15:00+00:00,4,2,376.39,517.12,140.73
5,DE_LU | price,PT15M,2025-10-01 17:00:00+00:00,4,2,408.50,499.00,90.50
6,DE_LU | price,PT15M,2025-10-01 16:45:00+00:00,4,2,306.90,381.00,74.10
7,DE_LU | price,PT15M,2025-10-01 09:00:00+00:00,4,2,83.50,149.92,66.42
8,DE_LU | price,PT15M,2025-10-01 05:45:00+00:00,4,2,170.54,234.70,64.16
9,DE_LU | price,PT15M,2025-10-01 16:00:00+00:00,4,2,146.95,206.00,59.05



per series:


,duplicated_stamps,redundant_copies,disagreeing,max_spread,first,last
series,,,,,,
DE_LU | price,10272,10464,192,278.69,2018-09-30 22:00:00+00:00,2025-10-02 21:45:00+00:00
DK1 | price,2880,2880,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00
DK2 | price,2880,2880,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00
NL | price,2880,2880,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00
NO2 | price,3384,3384,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00
SE3 | price,2880,2880,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00
SE4 | price,2880,2880,0,0.00,2016-01-31 23:00:00+00:00,2025-10-01 21:45:00+00:00



where the duplicated stamps sit in the month:


,duplicated_stamps
day_of_month,
1,26098
2,88
28,70
29,28
30,715
31,1057


In [ ]:
# Having been counted and characterised above, the redundant copies are dropped.
n = len(raw)
deduped = raw.drop_duplicates(KEY, keep="first").copy()

assert not deduped.duplicated(KEY).any(), "a duplicate key survived"
print(f"drop redundant copies: {n:,} -> {len(deduped):,}  (-{n - len(deduped):,})")

## 3. Do the PT15M quarters vary within the hour?

A `PT15M` stamp does not by itself mean quarter-hourly data. A zone that switched
its publication format before the auction cleared sub-hourly repeats one hourly
value across all four quarters, and a zone publishing a genuinely quarter-hourly
product does not.

The distinction decides what the collapse in section 4 does. Where the four values
of an hour are identical, the mean returns that value exactly. Where they differ,
the mean is a summary of four separate values.

Per series below: how many `PT15M` hours vary within the hour, the first hour that
varies, the largest within-hour spread, and the monthly share of varying hours.
Hours whose four values are all zero are counted separately as inactive.

In [ ]:
q = deduped.loc[deduped.resolution == "PT15M",
                ["series", "timestamp_utc", "value"]].copy()

if not len(q):
    print("no PT15M rows")
else:
    q["hour_utc"] = q.timestamp_utc.dt.floor("h")

    per_hour = (q.groupby(["series", "hour_utc"], observed=True)["value"]
                  .agg(slots="size", distinct="nunique", lo="min", hi="max"))
    per_hour["spread"] = per_hour["hi"] - per_hour["lo"]
    per_hour["varies"] = per_hour["distinct"] > 1
    # An all-zero hour cannot vary, so it is counted apart: solar at night would
    # otherwise drag the varying share down.
    per_hour["active"] = per_hour[["lo", "hi"]].abs().max(axis=1) > 1e-9
    ph = per_hour.reset_index()

    print(f"{len(ph):,} PT15M hours across {ph.series.nunique()} series | "
          f"{int(ph.varies.sum()):,} vary within the hour ({100 * ph.varies.mean():.2f}%)")

    active = ph[ph.active]
    first_varying = (ph[ph.varies].groupby("series", observed=True)["hour_utc"]
                       .min().rename("first_varying_hour"))
    pct_active = (100 * active.groupby("series", observed=True)["varies"].mean()
                  ).round(2).rename("pct_varying_active")

    summary = (ph.groupby("series", observed=True)
                 .agg(pt15_hours=("varies", "size"),
                      varying=("varies", "sum"),
                      inactive_hours=("active", lambda a: int((~a).sum())),
                      max_spread=("spread", "max"),
                      first_pt15=("hour_utc", "min"),
                      last_pt15=("hour_utc", "max"))
                 .join(first_varying)
                 .join(pct_active))
    summary["pct_varying"] = (100 * summary.varying / summary.pt15_hours).round(2)
    display(summary)

    print("\nmonthly share of active PT15M hours that vary within the hour:")
    display(active.assign(month=active.hour_utc.dt.tz_convert(None).dt.to_period("M"))
                  .pivot_table(index="month", columns="series", values="varies",
                               aggfunc="mean")
                  .round(3))

## 4. Collapse to the clock hour

Native resolutions differ across the seven zones. NL publishes its load and
generation forecasts at `PT15M` for its whole history and DE-LU from 2018; DK1,
DK2 and NO2 switch from `PT60M` to `PT15M` during 2025; SE3 and SE4 are hourly
throughout. This step puts them all on one hourly grid.

Two things are decided here, and each is reported:

1. **Sub-hourly values are averaged within the clock hour.** For an MW quantity
   that is the hourly average. For a price it is an unweighted mean of the
   quarters; the dump carries no traded volumes to weight by. Where section 3 shows
   the four quarters are identical the mean returns that value exactly. Hours
   arriving with fewer slots than their resolution implies are counted separately —
   the mean is then taken over part of an hour.

2. **An hour carrying both resolutions takes the `PT60M` value.** Several price
   series are published at both resolutions over overlapping spans; the `PT15M`
   mean fills only the hours `PT60M` does not cover. The disagreement between the
   two versions is measured and reported.

In [ ]:
# Slots per clock hour, which is also the guard: anything coarser than an hour would
# be relabelled PT60M by the collapse below. The weekly reservoir series is never
# read (section 1), so this only guards a future change to EXCLUDE_VARIABLES.
SLOTS_PER_HOUR = {"PT60M": 1, "PT15M": 4}

stray = sorted(set(deduped.resolution.unique()) - set(SLOTS_PER_HOUR))
assert not stray, f"resolution(s) {stray} are coarser than an hour"

# 1. collapse to the clock hour, per resolution, so a PT60M value is never averaged
# together with the PT15M quarters of the same hour
d = deduped.copy()
d["hour_utc"] = d.timestamp_utc.dt.floor("h")
GROUP = [c for c in ["zone", "eic", "variable", "document_type", "process_type",
                     "psr_type", "business_type", "curve_type", "contract_type",
                     "unit", "currency", "series", "resolution", "hour_utc"]
         if c in d.columns]

folded = (d.groupby(GROUP, as_index=False, observed=True)
            .agg(value=("value", "mean"), slots=("value", "size")))
print(f"1. collapse to the hour: {len(d):,} -> {len(folded):,}  (-{len(d) - len(folded):,})")

folded["expected_slots"] = folded.resolution.map(SLOTS_PER_HOUR)
partial = folded[folded.slots < folded.expected_slots]
print(f"   partial hours       : {len(partial):,} "
      f"({100 * len(partial) / len(folded):.4f}% of collapsed hours)")

In [ ]:
# 2. hours carrying both resolutions: how far do the two published versions differ?
overlap = folded.pivot_table(index=["series", "hour_utc"], columns="resolution",
                             values="value", aggfunc="first")
pair = [c for c in ("PT60M", "PT15M") if c in overlap.columns]

if len(pair) == 2:
    both = overlap.dropna(subset=pair).copy()
    both["abs_diff"] = (both["PT60M"] - both["PT15M"]).abs()
    print(f"hours published at both resolutions: {len(both):,}")
    if len(both):
        display(both.reset_index().groupby("series", observed=True)
                    .agg(hours=("abs_diff", "size"),
                         max_abs_diff=("abs_diff", "max"),
                         median_abs_diff=("abs_diff", "median"),
                         hours_differing=("abs_diff", lambda s: int((s > 1e-9).sum())),
                         first=("hour_utc", "min"), last=("hour_utc", "max")))
else:
    print("only one resolution present; nothing to reconcile")

In [ ]:
# PT60M wins where an hour carries both; the PT15M mean fills the rest.
PRECEDENCE = ["PT60M", "PT15M"]

folded["_rank"] = folded.resolution.map({r: i for i, r in enumerate(PRECEDENCE)})
hourly = (folded.sort_values(["series", "hour_utc", "_rank"])
                .drop_duplicates(["series", "hour_utc"], keep="first")
                .drop(columns="_rank")
                .rename(columns={"hour_utc": "timestamp_utc",
                                 "resolution": "source_resolution"})
                .reset_index(drop=True))

assert not hourly.duplicated(["series", "timestamp_utc"]).any(), \
    "a series still has two values for one hour"
assert hourly.timestamp_utc.dt.minute.eq(0).all(), "a stamp is not on the hour"

print(f"3. one row per series-hour: {len(folded):,} -> {len(hourly):,}  "
      f"(-{len(folded) - len(hourly):,})")
print(f"\n{len(hourly):,} rows | {hourly.series.nunique()} series | "
      f"{hourly.zone.nunique()} zones")
print(f"span: {hourly.timestamp_utc.min()} -> {hourly.timestamp_utc.max()} (UTC)")

display(hourly.groupby(["zone", "variable", "psr_type"], observed=True)
              .agg(hours=("value", "size"),
                   source=("source_resolution", lambda s: ",".join(sorted(s.unique()))),
                   first=("timestamp_utc", "min"),
                   last=("timestamp_utc", "max")))

## 5. Missing-value analysis

All series are now on the same hourly UTC grid. Three descriptive views; nothing
is filled, dropped or truncated here.

**Coverage**: hours published against hours spanned, between each series' own first
and last observation. A late start or an early stop shows in `first`/`last`, so
this measures interior holes. `partial_hours` counts hours that are present but
were collapsed from fewer slots than their resolution implies.

**Gap runs**: consecutive missing hours grouped into runs, with start, end and
length.

**Monthly coverage**: the share of each month's hours that were published, per
series.

In [5]:
def coverage(d: pd.DataFrame) -> pd.DataFrame:
    """Observed vs expected hours per series, between its own first and last stamp."""
    rows = []
    for s, g in d.groupby("series", observed=True):
        stamps = pd.DatetimeIndex(g.timestamp_utc.unique()).sort_values()
        grid = pd.date_range(stamps.min(), stamps.max(), freq="h")
        missing = len(grid) - len(stamps)
        rows.append({
            "series": s,
            "first": stamps.min(),
            "last": stamps.max(),
            "hours_spanned": len(grid),
            "observed": len(stamps),
            "missing": missing,
            "pct_missing": round(100 * missing / len(grid), 4),
            "partial_hours": int((g.slots < g.expected_slots).sum()),
        })
    return pd.DataFrame(rows).sort_values("series", ignore_index=True)


cov = coverage(hourly)
print(f"{int(cov.missing.sum()):,} missing hours across {len(cov)} series "
      f"({int((cov.missing > 0).sum())} series with at least one)")
display(cov)

44,442 missing hours across 33 series (21 series with at least one)


,series,first,last,hours_spanned,observed,missing,pct_missing,partial_hours
0,DE_LU | generation_forecast [solar],2018-09-30 22:00:00+00:00,2025-10-01 23:00:00+00:00,61394,61392,2,0.0033,0
1,DE_LU | generation_forecast [wind_offshore],2018-09-30 22:00:00+00:00,2025-10-01 23:00:00+00:00,61394,61394,0,0.0000,0
2,DE_LU | generation_forecast [wind_onshore],2018-09-30 22:00:00+00:00,2025-10-01 23:00:00+00:00,61394,61392,2,0.0033,0
3,DE_LU | load_forecast,2018-10-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,61392,60502,890,1.4497,0
4,DE_LU | price,2018-09-30 22:00:00+00:00,2025-10-02 21:00:00+00:00,61416,61416,0,0.0000,0
5,DK1 | generation_forecast [solar],2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,85488,85104,384,0.4492,0
6,DK1 | generation_forecast [wind_offshore],2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,85488,85248,240,0.2807,0
7,DK1 | generation_forecast [wind_onshore],2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,85488,85134,354,0.4141,1
8,DK1 | load_forecast,2016-01-01 00:00:00+00:00,2025-10-01 23:00:00+00:00,85488,85488,0,0.0000,0
9,DK1 | price,2015-12-31 23:00:00+00:00,2025-10-02 21:00:00+00:00,85511,85511,0,0.0000,0


In [ ]:
def gap_runs(d: pd.DataFrame) -> pd.DataFrame:
    """Every run of consecutive missing hours, per series."""
    runs = []
    for s, g in d.groupby("series", observed=True):
        stamps = pd.DatetimeIndex(g.timestamp_utc.unique()).sort_values()
        grid = pd.date_range(stamps.min(), stamps.max(), freq="h")
        missing = grid.difference(stamps)
        if not len(missing):
            continue
        gap = missing.to_series()
        breaks = (gap.diff() != pd.Timedelta(hours=1)).cumsum()
        for _, run in gap.groupby(breaks):
            runs.append({"series": s, "gap_start": run.index[0],
                         "gap_end": run.index[-1], "hours": len(run)})
    return pd.DataFrame(runs, columns=["series", "gap_start", "gap_end", "hours"])


runs = gap_runs(hourly)

if len(runs):
    print(f"{len(runs):,} gap runs, {int(runs.hours.sum()):,} missing hours in total\n")
    display(runs.groupby("series", observed=True)
                .agg(runs=("hours", "size"), missing_hours=("hours", "sum"),
                     longest_run=("hours", "max"), median_run=("hours", "median")))
    print("\n25 longest runs:")
    display(runs.nlargest(25, "hours").reset_index(drop=True))
else:
    print("no interior gaps in any series")

In [ ]:
def monthly_coverage(d: pd.DataFrame) -> pd.DataFrame:
    """Share of each month's hours that were published, per series."""
    out = {}
    for s, g in d.groupby("series", observed=True):
        stamps = pd.DatetimeIndex(g.timestamp_utc.unique()).sort_values()
        grid = pd.date_range(stamps.min(), stamps.max(), freq="h")
        have = pd.Series(grid.isin(stamps), index=grid)
        out[s] = have.groupby(grid.tz_convert(None).to_period("M")).mean()
    frame = pd.DataFrame(out).round(3)
    frame.index.name = "month"
    return frame


monthly = monthly_coverage(hourly)

# Only the months where something is actually missing, so the full-coverage runs do
# not bury them; the complete matrix stays in `monthly`.
incomplete = monthly[(monthly < 1).any(axis=1)]
print(f"{len(monthly)} months x {monthly.shape[1]} series | "
      f"{len(incomplete)} month(s) with at least one series below full coverage")
display(incomplete)